In [1]:
# ============================================================================
# CELL 1: Import và Setup
# ============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.models as models
from torchvision import transforms, datasets
import math
import numpy as np

print("Imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


Imports successful!
PyTorch version: 2.9.0+cu126
CUDA available: True


In [2]:
# ============================================================================
# CELL 2: APBLayer Class (Fixed inference_forward)
# ============================================================================
class APBLayer(nn.Module):
    def __init__(self, layer_to_wrap: nn.Module):
        super().__init__()
        if not isinstance(layer_to_wrap, (nn.Linear, nn.Conv2d)):
            raise ValueError("APBLayer chỉ hỗ trợ nn.Linear và nn.Conv2d.")

        self.wrapped_layer = layer_to_wrap
        self.latent_weight = nn.Parameter(layer_to_wrap.weight.data.clone())
        self.bias = layer_to_wrap.bias
        if hasattr(self.wrapped_layer, 'weight'):
            del self.wrapped_layer.weight

        with torch.no_grad():
            weights = self.latent_weight.data
            initial_alpha = weights.abs().mean()
            initial_delta = 3.0 * weights.std().clamp(min=1e-5)
        self.alpha = nn.Parameter(torch.tensor(initial_alpha.item(), device=weights.device))
        self.delta = nn.Parameter(torch.tensor(initial_delta.item(), device=weights.device))

    def forward(self, x):
        delta_clamped = self.delta.clamp(min=1e-8)
        w_hat = (self.latent_weight.abs() - self.alpha.abs()) / delta_clamped
        binarization_mask = (w_hat <= 1.0)
        sign_tensor = torch.sign(self.latent_weight)
        binarized_part = binarization_mask * sign_tensor * self.alpha.abs()
        full_precision_part = ~binarization_mask * self.latent_weight
        effective_weight = binarized_part + full_precision_part
        effective_weight = self.latent_weight + (effective_weight - self.latent_weight).detach()

        if isinstance(self.wrapped_layer, nn.Linear):
            return F.linear(x, effective_weight, self.bias)
        elif isinstance(self.wrapped_layer, nn.Conv2d):
            return F.conv2d(
                x, effective_weight, self.bias,
                self.wrapped_layer.stride, self.wrapped_layer.padding,
                self.wrapped_layer.dilation, self.wrapped_layer.groups
            )

    def get_stats(self):
        with torch.no_grad():
            total_weights = self.latent_weight.numel()
            threshold = self.alpha.abs() + self.delta.clamp(min=1e-8)
            num_binary = (self.latent_weight.abs() <= threshold).sum().item()
            return {
                "alpha": self.alpha.item(),
                "delta": self.delta.item(),
                "percent_binary": (num_binary / total_weights) * 100,
            }

    def get_effective_weight(self):
        delta_clamped = self.delta.clamp(min=1e-8)
        threshold = self.alpha.abs() + delta_clamped
        binarization_mask = self.latent_weight.abs() <= threshold
        sign_tensor = torch.ones_like(self.latent_weight)
        sign_tensor[self.latent_weight < 0] = -1
        binarized_part = torch.where(binarization_mask, 
                                   sign_tensor * self.alpha,
                                   torch.zeros_like(self.latent_weight))
        full_precision_part = torch.where(~binarization_mask, 
                                        self.latent_weight, 
                                        torch.zeros_like(self.latent_weight))
        return binarized_part + full_precision_part

print("APBLayer class defined!")

APBLayer class defined!


In [3]:
# ============================================================================
# CELL 3: Apply APB Function
# ============================================================================
def apply_apb(model: nn.Module, skip_first_conv=True, skip_downsample=True, skip_last_linear=True):
    conv_layers = [(name, module) for name, module in model.named_modules() if isinstance(module, nn.Conv2d)]
    first_conv_name = conv_layers[0][0] if conv_layers and skip_first_conv else None
    linear_layers = [(name, module) for name, module in model.named_modules() if isinstance(module, nn.Linear)]
    last_linear_name = linear_layers[-1][0] if linear_layers and skip_last_linear else None

    for name, module in list(model.named_modules()):
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            is_first_conv = (name == first_conv_name) and isinstance(module, nn.Conv2d)
            is_downsampling = 'downsample' in name and skip_downsample
            is_last_linear = (name == last_linear_name) and isinstance(module, nn.Linear)
            if is_first_conv or is_downsampling or is_last_linear:
                print(f"Skipping APB for: {name}")
                continue
            print(f"Applying APB to: {name}")
            parent_name = '.'.join(name.split('.')[:-1])
            child_name = name.split('.')[-1]
            parent_module = model
            if parent_name:
                for part in parent_name.split('.'):
                    parent_module = getattr(parent_module, part)
            setattr(parent_module, child_name, APBLayer(module))

    return model

print("apply_apb function defined!")


apply_apb function defined!


In [4]:
# ============================================================================
# CELL 4: Merge and Prune Functions
# ============================================================================
def l1inftyinfty(filter: torch.Tensor) -> float:
    abs_w = filter.abs()
    max_w = abs_w.max(dim=2).values
    max_hw = max_w.max(dim=1).values
    return max_hw.sum().item()

def l1inftyinfty_distance(filter1: torch.Tensor, filter2: torch.Tensor) -> float:
    diff = torch.abs(filter1 - filter2)
    return l1inftyinfty(diff)

def get_weight(layer: nn.Module):
    if isinstance(layer, APBLayer):
        return layer.get_effective_weight()
    elif isinstance(layer, (nn.Conv2d, nn.Linear)):
        return layer.weight
    else:
        raise ValueError("Layer phải là APBLayer, nn.Conv2d hoặc nn.Linear.")

def compute_distance_matrix(layer: nn.Module):
    if not isinstance(layer.wrapped_layer if isinstance(layer, APBLayer) else layer, nn.Conv2d):
        raise ValueError("compute_distance_matrix chỉ hỗ trợ Conv2d.")

    weight = get_weight(layer)
    out_channels = weight.shape[0]
    n = out_channels
    D = torch.zeros(n, n, device=weight.device)
    max_dist, min_dist, sum_dist, count = 0.0, float('inf'), 0.0, 0
    zero_dist_count = 0

    for i in range(n):
        for j in range(i + 1, n):
            filter1 = weight[i]
            filter2 = weight[j]
            dist = l1inftyinfty_distance(filter1, filter2)
            D[i, j] = dist
            D[j, i] = dist
            max_dist = max(max_dist, dist)
            min_dist = min(min_dist, dist)
            sum_dist += dist * 2
            count += 2
            if dist == 0:
                zero_dist_count += 1

    mean_dist = sum_dist / count if count > 0 else 0.0
    print(f"Distance matrix: min={min_dist:.4f}, max={max_dist:.4f}, mean={mean_dist:.4f}")
    print(f"Identical filter pairs: {zero_dist_count}")
    return D

def merge_two_filters(f1: torch.Tensor, f2: torch.Tensor, mode: str = 'arithmetic'):
    if f1.shape != f2.shape:
        raise ValueError("Hai filters phải có shape giống nhau.")

    if mode == 'arithmetic':
        return (f1 + f2) / 2
    elif mode == 'geometric':
        abs_prod = torch.sqrt(torch.abs(f1) * torch.abs(f2))
        sign = torch.sign(f1 + f2)
        return sign * abs_prod
    else:
        raise ValueError(f"Mode không hỗ trợ: {mode}")

def apply_merge(layer: nn.Module, threshold: float, mode: str = 'arithmetic'):
    if not isinstance(layer.wrapped_layer if isinstance(layer, APBLayer) else layer, nn.Conv2d):
        raise ValueError("apply_merge chỉ hỗ trợ Conv2d.")

    weight = get_weight(layer)
    D = compute_distance_matrix(layer)
    n = weight.shape[0]
    print(f"Số lượng filters: {n}")

    processed = set()
    merge_count = 0

    for i in range(n):
        if i in processed:
            continue
        for j in range(i + 1, n):
            if j in processed:
                continue
            if D[i, j] < threshold:
                merged = merge_two_filters(weight[i], weight[j], mode)
                weight[i] = merged
                weight[j] = torch.zeros_like(weight[j])
                processed.add(j)
                merge_count += 1

    if isinstance(layer, APBLayer):
        layer.latent_weight.data = weight.clone()

    print(f"Merged {merge_count} filter pairs")
    torch.cuda.empty_cache()
    return layer

print("Merge functions defined!")


Merge functions defined!


In [5]:
# ============================================================================
# CELL 5: Evaluation Function
# ============================================================================
def evaluate(model, data_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = running_loss / len(data_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

print("Evaluate function defined!")

Evaluate function defined!


In [6]:
# ============================================================================
# CELL 6: Main Training Script
# ============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load model
model_name = 'resnet18'
print(f"\nLoading pre-trained {model_name}...")
model = models.resnet18(weights='DEFAULT')
model.fc = nn.Linear(model.fc.in_features, 10)

# Apply APB
print("\nApplying APB...")
model_apb = apply_apb(model)
model_apb.to(device)
print("APB applied!")

# Data loaders
print("\nPreparing CIFAR-10 data...")
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# Training setup
optimizer = optim.Adam(model_apb.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
total_epochs = 10
params_frozen = False
save_path_before = '/kaggle/working/before_prune.pth'
save_path_after = '/kaggle/working/after_prune.pth'
best_accuracy = 0.0

print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)

Using device: cuda

Loading pre-trained resnet18...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 187MB/s]



Applying APB...
Skipping APB for: conv1
Applying APB to: layer1.0.conv1
Applying APB to: layer1.0.conv2
Applying APB to: layer1.1.conv1
Applying APB to: layer1.1.conv2
Applying APB to: layer2.0.conv1
Applying APB to: layer2.0.conv2
Skipping APB for: layer2.0.downsample.0
Applying APB to: layer2.1.conv1
Applying APB to: layer2.1.conv2
Applying APB to: layer3.0.conv1
Applying APB to: layer3.0.conv2
Skipping APB for: layer3.0.downsample.0
Applying APB to: layer3.1.conv1
Applying APB to: layer3.1.conv2
Applying APB to: layer4.0.conv1
Applying APB to: layer4.0.conv2
Skipping APB for: layer4.0.downsample.0
Applying APB to: layer4.1.conv1
Applying APB to: layer4.1.conv2
Skipping APB for: fc
APB applied!

Preparing CIFAR-10 data...


100%|██████████| 170M/170M [00:03<00:00, 47.7MB/s]



STARTING TRAINING


In [7]:
# ============================================================================
# CELL 7: Training Loop
# ============================================================================
for epoch in range(total_epochs):
    if epoch >= total_epochs // 2 and not params_frozen:
        print("\n" + "="*40)
        print(f"EPOCH {epoch}: Freezing alpha and delta")
        print("="*40)
        for module in model_apb.modules():
            if isinstance(module, APBLayer):
                module.alpha.requires_grad = False
                module.delta.requires_grad = False
        params_frozen = True

    model_apb.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_apb(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    test_loss, test_accuracy = evaluate(model_apb, test_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{total_epochs}] - "
          f"Train Loss: {train_loss:.4f}, "
          f"Test Loss: {test_loss:.4f}, "
          f"Test Acc: {test_accuracy:.2f}%")

    # Save best model (FIXED: use numpy.packbits)
    if test_accuracy > best_accuracy:
        print(f"→ New best! Saving to {save_path_before}")
        best_accuracy = test_accuracy
        state = model_apb.state_dict()
        
        for name, module in model_apb.named_modules():
            if isinstance(module, APBLayer):
                weight = module.get_effective_weight()
                threshold = module.alpha.abs() + module.delta.clamp(min=1e-8)
                fp_mask = weight.abs() > threshold
                fp_positions = torch.nonzero(fp_mask, as_tuple=False)
                fp_values = weight[fp_mask]
                full_signs = torch.sign(weight)
                sign_bits = (full_signs > 0).to(torch.uint8).view(-1)
                
                # FIX: Use numpy.packbits instead of torch.packbits
                sign_bits_cpu = sign_bits.cpu().numpy()
                packed_signs = torch.from_numpy(np.packbits(sign_bits_cpu))
                
                state[f"{name}.alpha"] = module.alpha
                state[f"{name}.packed_signs"] = packed_signs
                state[f"{name}.weight_shape"] = torch.tensor(weight.shape)
                state[f"{name}.fp_positions"] = fp_positions
                state[f"{name}.fp_values"] = fp_values
                
                if f"{name}.latent_weight" in state:
                    del state[f"{name}.latent_weight"]
                if f"{name}.delta" in state:
                    del state[f"{name}.delta"]
        
        torch.save(state, save_path_before)

    # Print stats every 5 epochs
    if (epoch + 1) % 5 == 0:
        try:
            example_layer = model_apb.layer1[0].conv1
            if isinstance(example_layer, APBLayer):
                stats = example_layer.get_stats()
                print(f"  Stats (layer1.0.conv1): "
                      f"α={stats['alpha']:.4f}, δ={stats['delta']:.4f}, "
                      f"Binary={stats['percent_binary']:.2f}%")
        except:
            pass

print("\n" + "="*60)
print("TRAINING COMPLETE")
print(f"Best accuracy before pruning: {best_accuracy:.2f}%")
print("="*60)

Epoch [1/10] - Train Loss: 0.5505, Test Loss: 0.3154, Test Acc: 89.41%
→ New best! Saving to /kaggle/working/before_prune.pth
Epoch [2/10] - Train Loss: 0.2397, Test Loss: 0.2586, Test Acc: 91.22%
→ New best! Saving to /kaggle/working/before_prune.pth
Epoch [3/10] - Train Loss: 0.1526, Test Loss: 0.2315, Test Acc: 92.35%
→ New best! Saving to /kaggle/working/before_prune.pth
Epoch [4/10] - Train Loss: 0.1071, Test Loss: 0.2489, Test Acc: 91.93%
Epoch [5/10] - Train Loss: 0.0851, Test Loss: 0.2472, Test Acc: 92.46%
→ New best! Saving to /kaggle/working/before_prune.pth
  Stats (layer1.0.conv1): α=0.0316, δ=0.1602, Binary=98.77%

EPOCH 5: Freezing alpha and delta
Epoch [6/10] - Train Loss: 0.0699, Test Loss: 0.2546, Test Acc: 92.39%
Epoch [7/10] - Train Loss: 0.0542, Test Loss: 0.2659, Test Acc: 92.02%
Epoch [8/10] - Train Loss: 0.0514, Test Loss: 0.2758, Test Acc: 92.15%
Epoch [9/10] - Train Loss: 0.0442, Test Loss: 0.2634, Test Acc: 92.44%
Epoch [10/10] - Train Loss: 0.0414, Test Loss: